# Item Dimension
This notebook builds `dim_items`, the reference table of every tradeable item in the game. The `/5m` price data
only gives item IDs, so this is what turns "4151" into "Abyssal whip" and tells you each item's
buy limit, which caps how much of it you can actually trade in a 4-hour window.

In [0]:
import pyspark.sql.functions as F

In [0]:
spark.sql('''USE CATALOG osrs_pipeline''')
spark.sql('''USE SCHEMA silver''')

## Read the mapping snapshot
`/mapping` returns a flat list of every item, so unlike the `/5m` data there's no nested map to
explode. Spark reads each list element straight into a row. Using `multiline` since the file is
pretty-printed JSON rather than one object per line.

In [0]:
MAPPING_PATH = "/Volumes/osrs_pipeline/bronze/item_mapping_raw/mapping.json"

In [0]:
mapping_df = spark.read.option("multiline", True).json(MAPPING_PATH)

## Select and cast
Only keeping the fields the flip logic actually needs: item ID, name, buy limit, and the members
flag. The rest (examine text, alch values, icon) don't feed anything downstream yet.

Two deliberate choices here:
- **`id` gets cast to string.** In the `/5m` data, item IDs arrive as JSON object *keys*, which are
  always strings, so `silver_prices.item_id` is a string. Here they're real numbers, so Spark infers
  a long. If I left them mismatched, the join between the fact table and this dimension would either
  fail or silently return nothing.
- **`limit` gets renamed to `buy_limit`.** `limit` is a reserved word in SQL, so leaving it would
  break any query that referenced it downstream.

In [0]:
dim_items = mapping_df.select(
    F.col("id").cast("string").alias("item_id"), 
    F.col("name"), 
    F.col("limit").alias("buy_limit"), 
    F.col("members")
    )

## Write the dimension
Overwriting the whole table rather than merging. `/mapping` returns the complete item list every
time, and item attributes only change when the game updates, so there's no history worth
accumulating and no need for the window-keyed idempotency the price pipeline uses. This is a
Type 1 dimension: current values only.

Note that `buy_limit` comes back null for some items. From what I can tell this means the limit
is undocumented rather than genuinely unlimited, so I leave it null rather than substituting a
value. Anything downstream that depends on it (like total profit per buy cycle) returns null too,
which is more honest than inventing a number I can't defend.

In [0]:
dim_items.write.format("delta").mode("overwrite").saveAsTable("osrs_pipeline.silver.dim_items")